In [ ]:

import requests
import pandas as pd

# 1. Descargar estructura PDB
def download_pdb(pdb_id):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url)
    with open(f"{pdb_id}.pdb", 'wb') as f:
        f.write(response.content)
    print(f"✅ Descargado {pdb_id}.pdb")

# 2. Obtener info de AlphaFold
def get_alphafold_prediction(uniprot_id):
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"AF_{uniprot_id}.pdb", 'wb') as f:
            f.write(response.content)
        print(f"✅ Descargada predicción AlphaFold para {uniprot_id}")
    return response.status_code == 200

# 3. Calcular métricas básicas
def analyze_mutation(pdb_id, mutations, fragment_size):
    msa_depth = max(10, 80 - len(mutations) * 12)  # Simulación
    qubits_needed = int(2.5 * fragment_size**2 + 10)
    
    score = 0
    if fragment_size <= 22: score += 40
    if msa_depth < 60: score += 30
    if 1 <= len(mutations) <= 15: score += 30
    
    return {
        "pdb_id": pdb_id,
        "mutations": len(mutations),
        "mutations_list": ", ".join(mutations),
        "fragment_size": fragment_size,
        "msa_depth": msa_depth,
        "qubits_needed": qubits_needed,
        "quantum_score": score,
        "recommendation": "QUANTUM" if score >= 70 else "CLASSICAL"
    }

# 4. Exploratorio de PDB (EDA)
def parse_pdb_atoms(pdb_id):
    atoms = []
    try:
        with open(f"{pdb_id}.pdb", "r") as f:
            for line in f:
                if line.startswith("ATOM") or line.startswith("HETATM"):
                    record = line[0:6].strip()
                    atom_name = line[12:16].strip()
                    res_name = line[17:20].strip()
                    chain_id = line[21].strip()
                    res_seq = line[22:26].strip()
                    try:
                        x = float(line[30:38])
                        y = float(line[38:46])
                        z = float(line[46:54])
                    except ValueError:
                        x = y = z = None
                    try:
                        b = float(line[60:66])
                    except ValueError:
                        b = None
                    atoms.append({
                        "record": record,
                        "atom_name": atom_name,
                        "res_name": res_name,
                        "chain_id": chain_id,
                        "res_seq": res_seq,
                        "x": x, "y": y, "z": z,
                        "bfactor": b,
                        "is_het": record == "HETATM",
                        "is_ca": atom_name == "CA"
                    })
    except FileNotFoundError:
        return []
    return atoms

def pdb_secondary_counts(pdb_id):
    helix = 0
    sheet = 0
    try:
        with open(f"{pdb_id}.pdb", "r") as f:
            for line in f:
                if line.startswith("HELIX"):
                    helix += 1
                elif line.startswith("SHEET"):
                    sheet += 1
    except FileNotFoundError:
        pass
    return helix, sheet

def pdb_stats_from_atoms(atoms):
    if not atoms:
        return {
            "atoms_count": 0,
            "hetatm_count": 0,
            "chains": "",
            "chains_count": 0,
            "residues_count": 0,
            "unique_residues_types": 0,
            "mean_bfactor": None,
            "std_bfactor": None,
            "bbox_x": None, "bbox_y": None, "bbox_z": None,
            "radius_gyration": None,
            "helix_count": None,
            "sheet_count": None
        }
    chains = sorted(set(a["chain_id"] for a in atoms if a["chain_id"]))
    res_keys = sorted(set((a["chain_id"], a["res_seq"]) for a in atoms))
    res_types = sorted(set(a["res_name"] for a in atoms if a["res_name"]))
    bfactors = [a["bfactor"] for a in atoms if a["bfactor"] is not None]
    xs = [a["x"] for a in atoms if a["x"] is not None]
    ys = [a["y"] for a in atoms if a["y"] is not None]
    zs = [a["z"] for a in atoms if a["z"] is not None]
    # Bounding box
    bbox_x = (min(xs), max(xs)) if xs else (None, None)
    bbox_y = (min(ys), max(ys)) if ys else (None, None)
    bbox_z = (min(zs), max(zs)) if zs else (None, None)
    # Radio de giro usando CA si existen
    ca_positions = [(a["x"], a["y"], a["z"]) for a in atoms if a["is_ca"] and a["x"] is not None]
    pos = ca_positions if ca_positions else [(a["x"], a["y"], a["z"]) for a in atoms if a["x"] is not None]
    if pos:
        cx = sum(p[0] for p in pos) / len(pos)
        cy = sum(p[1] for p in pos) / len(pos)
        cz = sum(p[2] for p in pos) / len(pos)
        import math
        rg = math.sqrt(sum((p[0]-cx)**2 + (p[1]-cy)**2 + (p[2]-cz)**2 for p in pos) / len(pos))
    else:
        rg = None
    return {
        "atoms_count": len(atoms),
        "hetatm_count": sum(1 for a in atoms if a["is_het"]),
        "chains": ", ".join(chains),
        "chains_count": len(chains),
        "residues_count": len(res_keys),
        "unique_residues_types": len(res_types),
        "mean_bfactor": round(sum(bfactors)/len(bfactors), 3) if bfactors else None,
        "std_bfactor": round(pd.Series(bfactors).std(), 3) if bfactors else None,
        "bbox_x": bbox_x,
        "bbox_y": bbox_y,
        "bbox_z": bbox_z,
        "radius_gyration": round(rg, 3) if rg is not None else None
    }

def analyze_pdb(pdb_id):
    atoms = parse_pdb_atoms(pdb_id)
    stats = pdb_stats_from_atoms(atoms)
    h, s = pdb_secondary_counts(pdb_id)
    stats["pdb_id"] = pdb_id
    stats["helix_count"] = h
    stats["sheet_count"] = s
    return stats

# TEST RÁPIDO
if __name__ == "__main__":
    # Caso de prueba: β-lactamasa
    download_pdb("1BTL")
    download_pdb("1RX2")
    download_pdb("1KZN")
    
    results_1BTL = analyze_mutation(
        pdb_id="1BTL",
        mutations=["E104K"],
        fragment_size=20
    )

    results_1RX2 = analyze_mutation(
        pdb_id="1RX2",
        mutations=["F98Y"],
        fragment_size=20
    )

    results_1KZN = analyze_mutation(
        pdb_id="1KZN",
        mutations=["S83L", "D87N"],
        fragment_size=20
    )
    
    print("\n📊 ANÁLISIS 1BTL/E104K:")
    for key, value in results_1BTL.items():
        print(f"  {key}: {value}")

    print("\n📊 ANÁLISIS 1RX2/F98Y:")
    for key, value in results_1RX2.items():
        print(f"  {key}: {value}")

    print("\n📊 ANÁLISIS 1KZN/S83L,D87N:")
    for key, value in results_1KZN.items():
        print(f"  {key}: {value}")
    
    # Crear DataFrame para visualización
    df = pd.DataFrame([results_1BTL, results_1RX2, results_1KZN])
    from IPython.display import display
    print("\nFilas esperadas: 3. Forma:", df.shape)
    display(df)
    print("\nVista completa:")
    print(df.to_string(index=False))

    # EDA por PDB: estadísticas estructurales
    pdb_stats_list = [analyze_pdb("1BTL"), analyze_pdb("1RX2"), analyze_pdb("1KZN")]
    df_pdb = pd.DataFrame(pdb_stats_list)
    print("\nPDB stats shape:", df_pdb.shape)
    display(df_pdb)

    # Unir ambos DataFrames por pdb_id
    df_combined = pd.merge(df, df_pdb, on="pdb_id", how="left")
    print("\nCombined shape:", df_combined.shape)
    display(df_combined)
    df.to_csv("results.csv", index=False)
    df_pdb.to_csv("pdb_stats.csv", index=False)
    df_combined.to_csv("combined_results.csv", index=False)




✅ Descargado 1BTL.pdb
✅ Descargado 1RX2.pdb
✅ Descargado 1KZN.pdb

📊 ANÁLISIS 1BTL/E104K:
  pdb_id: 1BTL
  mutations: 1
  mutations_list: E104K
  fragment_size: 20
  msa_depth: 68
  qubits_needed: 1010
  quantum_score: 70
  recommendation: QUANTUM

📊 ANÁLISIS 1RX2/F98Y:
  pdb_id: 1RX2
  mutations: 1
  mutations_list: F98Y
  fragment_size: 20
  msa_depth: 68
  qubits_needed: 1010
  quantum_score: 70
  recommendation: QUANTUM

📊 ANÁLISIS 1KZN/S83L,D87N:
  pdb_id: 1KZN
  mutations: 2
  mutations_list: S83L, D87N
  fragment_size: 20
  msa_depth: 56
  qubits_needed: 1010
  quantum_score: 100
  recommendation: QUANTUM

Filas esperadas: 3. Forma: (3, 8)


,pdb_id,mutations,mutations_list,fragment_size,msa_depth,qubits_needed,quantum_score,recommendation
0,1BTL,1,E104K,20,68,1010,70,QUANTUM
1,1RX2,1,F98Y,20,68,1010,70,QUANTUM
2,1KZN,2,"S83L, D87N",20,56,1010,100,QUANTUM



Vista completa:
pdb_id  mutations mutations_list  fragment_size  msa_depth  qubits_needed  quantum_score recommendation
  1BTL          1          E104K             20         68           1010             70        QUANTUM
  1RX2          1           F98Y             20         68           1010             70        QUANTUM
  1KZN          2     S83L, D87N             20         56           1010            100        QUANTUM

PDB stats shape: (3, 15)


,atoms_count,hetatm_count,chains,chains_count,residues_count,unique_residues_types,mean_bfactor,std_bfactor,bbox_x,bbox_y,bbox_z,radius_gyration,pdb_id,helix_count,sheet_count
0,2236,204,A,1,463,22,11.294,6.622,"(-16.263, 28.5)","(-20.185, 22.476)","(3.215, 61.722)",17.684,1BTL,12,5
1,1503,235,A,1,316,25,21.950,16.157,"(7.79, 48.284)","(21.445, 68.196)","(-7.125, 34.05)",15.152,1RX2,4,8
2,1659,213,A,1,351,22,20.705,7.848,"(-0.911, 45.553)","(-3.56, 43.015)","(23.291, 64.325)",15.771,1KZN,6,16



Combined shape: (3, 22)


,pdb_id,mutations,mutations_list,fragment_size,msa_depth,qubits_needed,quantum_score,recommendation,atoms_count,hetatm_count,...,residues_count,unique_residues_types,mean_bfactor,std_bfactor,bbox_x,bbox_y,bbox_z,radius_gyration,helix_count,sheet_count
0,1BTL,1,E104K,20,68,1010,70,QUANTUM,2236,204,...,463,22,11.294,6.622,"(-16.263, 28.5)","(-20.185, 22.476)","(3.215, 61.722)",17.684,12,5
1,1RX2,1,F98Y,20,68,1010,70,QUANTUM,1503,235,...,316,25,21.950,16.157,"(7.79, 48.284)","(21.445, 68.196)","(-7.125, 34.05)",15.152,4,8
2,1KZN,2,"S83L, D87N",20,56,1010,100,QUANTUM,1659,213,...,351,22,20.705,7.848,"(-0.911, 45.553)","(-3.56, 43.015)","(23.291, 64.325)",15.771,6,16
